# ✅ 1. 구글 드라이브 마운트 (Colab 환경)

In [ ]:
# Google Drive 연동
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ✅ 2. 라이브러리 설치 및 불러오기

### 1) 설치

In [ ]:
pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.0 MB/s eta 0:00:00


### 2)  불러오기

In [ ]:
# 기본 라이브러리
import os
import re
import torch
import numpy as np
import pandas as pd
from glob import glob
from tqdm.auto import tqdm
from IPython.display import Markdown, display

# PDF 처리
import PyPDF2

# 텍스트 벡터화 및 유사도 분석
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 자연어 처리 모델
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# ✅ 3. 데이터 로드 및 전처리

In [ ]:
# 데이터 로드
train = pd.read_csv("/content/drive/MyDrive/건설공사 사고 예방 및 대응책 생성 한솔데코/train.csv")
test = pd.read_csv("/content/drive/MyDrive/건설공사 사고 예방 및 대응책 생성 한솔데코/test.csv")
sample = pd.read_csv("/content/drive/MyDrive/건설공사 사고 예방 및 대응책 생성 한솔데코/sample_submission.csv")

# ✅ 4. 대책 요약 생성

In [ ]:

# ✅ 1. 텍스트 전처리 및 문장 분리 함수
# 텍스트에서 불필요한 공백과 개행 제거(텍스트 전처리)
def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)  # 다중 공백 제거
    text = re.sub(r'\n', ' ', text)   # 줄바꿈 제거
    return text.strip()

# 텍스트를 문장 단위로 분리
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)



# ✅ 2. Markdown 파일 병합 및 문장화
# Markdown 파일 처리 함수 (PDF 대신 사용)
# (page_1.md, page_2.md, ...)을 읽어 하나의 문자열로 결합
def read_markdown_files(directory):
    page_num = 1
    combined_text = ""

    while True:
        page_path = os.path.join(directory, f"page_{page_num}.md")
        if not os.path.exists(page_path):
            break

        with open(page_path, "r", encoding="utf-8") as f:
            combined_text += f.read() + "\n"

        page_num += 1

    return combined_text



# ✅ 3. SBERT 임베딩 + TF-IDF 유사도 기반 대표 문장 추출
def prepare_similarities(sentences, embeddings):
    with tqdm(total=2, desc="유사도 계산", leave=False) as pbar:

        # 문맥 유사도 (임베딩 기반)
        pbar.set_postfix_str("문맥 유사도")
        context_sim = cosine_similarity(embeddings)
        pbar.update(1)

        # 단어 기반 유사도 (TF-IDF)
        pbar.set_postfix_str("단어 유사도")
        tfidf = TfidfVectorizer().fit_transform(sentences)
        word_sim = cosine_similarity(tfidf)
        pbar.update(1)

    return context_sim, word_sim

# 두 유사도를 가중 평균하여 최종 유사도 계산
def combined_similarity(context_sim, word_sim, alpha=0.6):
    return alpha * context_sim + (1 - alpha) * word_sim



# ✅ 4. 메인 파이프라인: 사고 유형별 요약 생성

def main():
    # ✔️ 4-1. 시스템 초기화 및 모델 로딩
    print("🚀 시스템 초기화 중...")
    folder_path = "/content/drive/MyDrive/DACON/건설안전지침_md"  # Markdown 폴더 경로
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✔️ 사용 중인 장치: {device}")
    print("🔧 모델 로드 중...")

    # SBERT 모델 (문장 임베딩)
    model = SentenceTransformer("jhgan/ko-sbert-sts").to(device)

    # LLM 모델 (요약 생성)
    llm_model = AutoModelForCausalLM.from_pretrained(
        "upstage/SOLAR-10.7B-Instruct-v1.0",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(
        "upstage/SOLAR-10.7B-Instruct-v1.0",
        padding_side="left",
        pad_token="<|endoftext|>"
    )

    # 데이터 로드
    print("📂 데이터 준비 중...")
    train  # train 데이터프레임이 이미 로드되어 있어야 함.


    # ✔️ 4-2. 사고 유형별 그룹화
    grouped = train.groupby("인적사고")
    total_groups = len(grouped)
    print(f" 총 처리 그룹 수: {total_groups}")

    res = {} # 대표 문장 저장
    res_enhanced = {} # 요약 결과 저장
    cosine_res = []

     # ✔️ 4-3. 그룹별 문장 수집 및 전처리
    with tqdm(total=total_groups, desc="🏗️ 전체 그룹 처리", bar_format="{l_bar}{bar:20}{r_bar}", position=0) as main_pbar:
        for name, group in grouped:
            main_pbar.set_postfix_str(f"현재 그룹: {name}")

            # 기본 데이터 처리
            with tqdm(total=5, desc=f"🔍 {name} 분석", leave=False, position=1) as group_pbar:
                # 기본 문장 추출
                group_pbar.set_description("[1/5] 기본 데이터 수집")
                plan = group["재발방지대책 및 향후조치계획"]
                sentences = plan.tolist()
                group_pbar.update(1)

                # Markdown 파일 처리 (PDF 대신 사용)
                group_pbar.set_description("[2/5] Markdown 파일 분석")
                # Markdown 문서 문장 추가
                markdown_dirs = glob(os.path.join(folder_path, "*"))  # 모든 하위 디렉토리 검색

                for markdown_dir in markdown_dirs:
                    markdown_text = read_markdown_files(markdown_dir)
                    if markdown_text:
                        processed_text = preprocess_text(markdown_text)
                        markdown_sentences = split_sentences(processed_text)
                        sentences.extend(markdown_sentences)

                group_pbar.update(1)

                # 임베딩 생성
                group_pbar.set_description("[3/5] 임베딩 생성")
                # 문장 임베딩
                vectors = model.encode(
                    sentences,
                    batch_size=32,
                    show_progress_bar=False,
                    device=device
                )
                group_pbar.update(1)


                # ✔️ 4-4. 문장 임베딩 및 유사도 분석
                group_pbar.set_description("[4/5] 유사도 분석")
                # 유사도 계산 후 대표 문장 선택
                context_sim, word_sim = prepare_similarities(sentences, vectors)
                combined_sim = combined_similarity(context_sim, word_sim)
                best_idx = np.argmax(np.mean(combined_sim, axis=1))
                group_pbar.update(1)

                # AI 요약 생성
                group_pbar.set_description("[5/5] AI 요약 생성")
                representative_plan = sentences[best_idx]
                res[name] = representative_plan

                rag_prompt = f"""
                ### 지침: 당신은 건설 안전 전문가입니다.
                - 질문에 대한 답변을 핵심 내용만 요약하여 간략하게 작성하세요.
                - 단어 및 문장 강조를 위한 "**" 표시를 절대 포함하지 마세요.
                - 서론, 배경 설명 또는 추가 설명을 절대 포함하지 마세요.
                - 특수기호 (=-)를 포함하지 마세요.
                - 다음과 같은 조치를 취할 것을 제안합니다: 와 같은 내용을 포함하지 마세요.
                - ### 제안 사항 요약:, ### 핵심 조치 요약: 와 같은 내용을 포함하지 마세요.

                {representative_plan}"""

                # 채팅 형식 입력
                messages = [
                    {"role": "system", "content": "전달받은 내용을 단 한 글자도 바꾸지 않고 완전히 그대로 출력합니다."},
                    {"role": "user", "content": rag_prompt}
                ]

                 # 모델 입력 구성
                inputs = tokenizer.apply_chat_template(
                    messages,
                    add_generation_prompt=True,
                    return_tensors="pt",
                    return_attention_mask=True,
                    return_dict=True
                    ).to(llm_model.device)

                  # 입력 분리
                input_ids = inputs["input_ids"]
                attention_mask = inputs["attention_mask"]

                # LLM 응답 생성
                output = llm_model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=256,
                    do_sample=True,
                    temperature=0.7,
                    top_k=20,
                    top_p=0.85
                    )

                # ✔️ 4-5. 대표 문장 기반 요약 생성 (LLM)
                generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
                assistant_response = re.sub(r"### (System|Assistant):.*?", "", generated_text, flags=re.DOTALL).strip()  # LLM 응답에서 시스템 메시지 제거
                match = re.search(r"### User:\s*(.*?)$", assistant_response, flags=re.DOTALL)  # "### User:" 이후의 텍스트만 추출
                res_enhanced[name] = match.group(1).strip() if match else assistant_response  # 최종 요약 결과 저장

            main_pbar.update(1)

    print("\n🎉 모든 처리 완료!")
    # 최종 결과 반환 추가
    return res_enhanced, model


# ✅ 5. 실행 및 결과 저장
if __name__ == "__main__":
  res_enhanced, model = main()  # 결과 저장


🚀 시스템 초기화 중...
✔️ 사용 중인 장치: cuda
🔧 모델 로드 중...


config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.8k [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

📂 데이터 준비 중...
✔️ 총 처리 그룹 수: 23


[4/5] 유사도 분석:  60%|██████    | 3/5 [00:00<00:00, 39.08it/s] 

유사도 계산:   0%|          | 0/2 [00:00<?, ?it/s]

유사도 계산:   0%|          | 0/2 [00:00<?, ?it/s, 문맥 유사도]

유사도 계산:  50%|█████     | 1/2 [00:00<00:00, 366.28it/s, 단어 유사도]

                                                                        
[4/5] 유사도 분석:  60%|██████    | 3/5 [00:00<00:00, 34.04it/s] 

유사도 계산:   0%|          | 0/2 [00:00<?, ?it/s]

유사도 계산:   0%|          | 0/2 [00:00<?, ?it/s, 문맥 유사도]

유사도 계산:  50%|█████     | 1/2 [00:00<00:00, 263.81it/s, 단어 유사도]

                                                                        
[4/5] 유사도 분석:  60%|██████    | 3/5 [00:01<00:00,  2.89it/s]

유사도 계산:   0%|          | 0/2 [00:00<?, ?it/s]

유사도 계산:   0%|          | 0/2 [00:00<?, ?it/s, 문맥 유사도]

유사도 계산:  50%|█████     | 1/2 [00:00<00:00, 39.05it/s, 단어 유사도]

유사도 계산: 100%|██████████| 2/2 [00:00<00:00, 11.89it/s, 단어 유사도]

                                                                       
[4/5] 유사도 분석:  60%|██████    | 3/5 [0


🎉 모든 처리 완료!


# ✅ 5. 테스트셋 요약 결과 반영 및 저장

In [ ]:
# 테스트셋에 예측 결과 채우기
for i in tqdm(range(len(test))):
    accident = test.loc[i, "인적사고"]
    if accident in res_enhanced:
        sample.loc[i, "재발방지대책 및 향후조치계획"] = res_enhanced[accident]
        sample.iloc[i, 2:] = model.encode(res_enhanced[accident])

# 결과 저장
sample.to_csv("baseline.csv", index=False, encoding="utf-8-sig")

100%|██████████| 964/964 [01:40<00:00,  9.64it/s]


# ✅ 6. 후처리 및 정제 임베딩 재생성

In [ ]:
# 후처리 및 벡터 재생성
embedding_model = SentenceTransformer("jhgan/ko-sbert-sts")

# CSV 파일 로드
file_path = "baseline.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

# 불필요한 문장/기호 제거
def clean_text(text):
    if pd.isna(text):
        return ""

    # 프롬프트 관련 불필요한 문구 제거
    text = re.sub(r"### 지침:.*?\n", "", text, flags=re.DOTALL)  # 전체 프롬프트 제거
    text = re.sub(r"- .*?\n", "", text)  # 불필요한 가이드라인 제거
    text = re.sub(r"### 제안 사항 요약:.*?\n", "", text)  # 요약 가이드 제거
    text = re.sub(r"### 핵심 조치 요약:.*?\n", "", text)  # 핵심 조치 제거

    # 영어 문장 자동 제거 (한국어 문장만 남기기)
    text = re.sub(r"[A-Za-z].*\n?", "", text)  # 영어 문장 제거 (한 줄씩 탐색하여 삭제)

    # 특수기호 제거 (=- 등 불필요한 기호 제거)
    text = re.sub(r"[-=]{2,}", "", text)

    # 숫자 리스트 제거 (예: 1. 2. 3. 4. 등)
    text = re.sub(r"\n?\d+\.\s+", "", text)  # 숫자와 점(`.`) 뒤의 공백까지 삭제

    # 공백 및 개행 정리
    text = text.strip()
    text = re.sub(r"\n+", "\n", text)  # 여러 개의 개행을 하나로 축소

    return text


# 텍스트 정제
df["재발방지대책 및 향후조치계획"] = df["재발방지대책 및 향후조치계획"].apply(clean_text)

# # 임베딩 재생성 (정규화 포함), 후처리된 답변 벡터화
cleaned_texts = df["재발방지대책 및 향후조치계획"].tolist()
embeddings = embedding_model.encode(cleaned_texts, normalize_embeddings=True)  # 벡터 정규화

# 벡터를 DataFrame에 삽입, vec_0부터 갱신
embedding_df = pd.DataFrame(embeddings, columns=[f"vec_{i}" for i in range(embeddings.shape[1])])
df.iloc[:, 2:] = embedding_df.values

# 최종 CSV 저장
output_file_path = "/content/drive/MyDrive/DACON/0324_baseline.csv"
df.to_csv(output_file_path, index=False, encoding="utf-8-sig")


In [ ]:
# 최종 csv 파일 확인

pd.set_option('display.max_colwidth', None)

# 데이터 프레임 출력
df['재발방지대책 및 향후조치계획'].head(30)


,재발방지대책 및 향후조치계획
0,작업 전 안전교육 실시와 안전조치 강화를 통한 재발 방지 대책 마련.
1,안전교육 실시와 작업 시 안전관리 철저를 통한 재발 방지 대책 및 향후 조치 계획.
2,작업 전 안전교육 실시를 통한 재발 방지 대책 및 향후 조치 계획.
3,현장정리정돈 실시와 작업 전 안전교육 철저를 통한 재발 방지 대책 및 향후 조치 계획.
4,작업 전 안전교육 실시를 통한 재발 방지 대책 및 향후 조치 계획.
5,작업자의 안전교육 실시와 현장 안전관리 철저 지시 및 안전 점검 실시를 통한 재발 방지 대책.
6,"안전교육 실시와 작업자 안전관리 철저, 안전시설 점검을 통한 재발 방지 대책 마련."
7,작업자의 안전교육 실시와 현장 안전관리 철저 지시 및 안전 점검 실시를 통한 재발 방지 대책.
8,안전교육 실시와 작업 중 안전관리 철저를 통한 재발 방지 대책 및 향후 조치 계획.
9,작업 전 안전교육 실시와 안전조치 강화를 통한 재발 방지 대책 마련.


# ✅ 7. 텍스트 후처리 함수

In [ ]:
# 텍스트 후처리 함수 정의
def clean_text_strict(text):
    if pd.isna(text):
        return ""

    # 기본 정리
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    # 가장 마지막 '다.' 기준 자르기
    last_sentence_end = text.rfind("다.")
    if last_sentence_end != -1:
        return text[:last_sentence_end + 2]

    # 없으면 '요.'로 자르기
    last_sentence_end = text.rfind("요.")
    if last_sentence_end != -1:
        return text[:last_sentence_end + 2]

    # 그래도 없으면 마침표로 자르기
    last_period = text.rfind(".")
    if last_period != -1:
        return text[:last_period + 1]

    return text  # fallback


In [ ]:
# 텍스트 정제 적용
df["재발방지대책 및 향후조치계획"] = df["재발방지대책 및 향후조치계획"].apply(clean_text_strict)

# 임베딩 모델 로드
embedding_model = SentenceTransformer("jhgan/ko-sbert-sts")

# 벡터화
texts = df["재발방지대책 및 향후조치계획"].tolist()
embeddings = embedding_model.encode(texts, normalize_embeddings=True)

In [ ]:
# 벡터를 DataFrame으로 변환
embedding_df = pd.DataFrame(embeddings, columns=[f"vec_{i}" for i in range(embeddings.shape[1])])

# 기존 df에 벡터 붙이기
df = pd.concat([df.drop(columns=[col for col in df.columns if col.startswith("vec_")]), embedding_df], axis=1)


In [ ]:
# 저장
output_path = "0324_baseline_cleaned.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")